In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime, timedelta
from delta.tables import DeltaTable


# ── CONFIG ────────────────────────────────────────────────────────────────────
GOLD_TABLE = "telecom.gold.gold_customer_telecom"
today = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
# ─────────────────────────────────────────────────────────────────────────────

print(f"{'='*60}")
print(f"Gold notebook started")
print(f"Processing date: {today}")
print(f"{'='*60}")

# ── Step 1: Read only today's partition from silver ───────────────────────────
silver_cdr = (
    spark.read.table("telecom.silver.silver_cdr")
        .filter(col("call_date") == today)
)

silver_events = (
    spark.read.table("telecom.silver.silver_network_events")
        .filter(col("event_date") == today)
)

cdr_count    = silver_cdr.count()
events_count = silver_events.count()

print(f"\nSilver rows for {today}:")
print(f"  CDR rows:    {cdr_count:,}")
print(f"  Events rows: {events_count:,}")

if cdr_count == 0:
    raise Exception(f"No CDR data found for {today}!")

if events_count == 0:
    raise Exception(f"No Events data found for {today}!")

# ── Step 2: Aggregate CDR per customer ───────────────────────────────────────
print(f"\nAggregating CDR per customer...")
cdr_agg = (
    silver_cdr
        .groupBy("customer_id")
        .agg(
            count("cdr_id")                             .alias("total_calls"),
            sum("call_duration_mins")                   .alias("total_call_mins"),
            round(avg("call_duration_mins"), 2)         .alias("avg_call_duration"),
            round(sum("charge"), 4)                     .alias("total_charges"),
            sum("is_dropped").cast(LongType())          .alias("dropped_calls"),
            sum("is_missed").cast(LongType())           .alias("missed_calls"),
            sum("is_international").cast(LongType())    .alias("international_calls"),
            sum("is_peak_hour").cast(LongType())        .alias("peak_hour_calls"),
            round(avg("revenue_per_minute"), 4)         .alias("avg_revenue_per_min")
        )
)

# ── Step 3: Aggregate events per customer ────────────────────────────────────
print(f"Aggregating events per customer...")
events_agg = (
    silver_events
        .groupBy("customer_id")
        .agg(
            count("event_id")                           .alias("total_events"),
            sum("is_dropped_call").cast(LongType())     .alias("network_drops"),
            sum("is_outage").cast(LongType())           .alias("tower_outages"),
            sum("is_critical").cast(LongType())         .alias("critical_events"),
            round(avg("signal_strength"), 2)            .alias("avg_signal_strength"),
            round(avg("data_speed_mbps"), 2)            .alias("avg_data_speed"),
            countDistinct("tower_id").cast(LongType())  .alias("towers_used")
        )
)

# ── Step 4: Join + derive churn metrics ──────────────────────────────────────
print(f"Joining CDR + Events and calculating churn metrics...")
gold_df = (
    cdr_agg
        .join(events_agg, on="customer_id", how="left")
        .fillna(0, subset=[
            "network_drops", "tower_outages",
            "critical_events", "total_events",
            "avg_signal_strength", "avg_data_speed", "towers_used"
        ])
        .withColumn("drop_rate_pct",
                    round(
                        col("dropped_calls").cast(DoubleType()) /
                        col("total_calls").cast(DoubleType()) * 100.0, 2))
        .withColumn("missed_rate_pct",
                    round(
                        col("missed_calls").cast(DoubleType()) /
                        col("total_calls").cast(DoubleType()) * 100.0, 2))
        .withColumn("avg_charge_per_call",
                    round(
                        col("total_charges").cast(DoubleType()) /
                        col("total_calls").cast(DoubleType()), 4))
        .withColumn("churn_risk_score",
                    round(
                        (col("drop_rate_pct").cast(DoubleType())   * 0.4) +
                        (col("critical_events").cast(DoubleType()) * 0.3) +
                        (col("network_drops").cast(DoubleType())   * 0.3), 2))
        .withColumn("churn_risk_label",
                    when(col("churn_risk_score") >= 15, "high")
                    .when(col("churn_risk_score") >= 7,  "medium")
                    .otherwise("low"))
        .withColumn("is_high_value",
                    when(col("total_charges") >= 50, lit(1)).otherwise(lit(0)))
        .withColumn("network_health_score",
                    round(
                        lit(100.0) -
                        (col("network_drops").cast(DoubleType())   * 2.0) -
                        (col("critical_events").cast(DoubleType()) * 5.0), 2))
        .withColumn("snapshot_date",       current_date())
        .withColumn("pipeline_updated_at", current_timestamp())
        .select(
            col("snapshot_date"),
            col("customer_id"),
            col("total_calls").cast(LongType())             .alias("total_calls"),
            col("total_call_mins").cast(DoubleType())       .alias("total_call_mins"),
            col("avg_call_duration").cast(DoubleType())     .alias("avg_call_duration"),
            col("total_charges").cast(DoubleType())         .alias("total_charges"),
            col("dropped_calls").cast(LongType())           .alias("dropped_calls"),
            col("missed_calls").cast(LongType())            .alias("missed_calls"),
            col("international_calls").cast(LongType())     .alias("international_calls"),
            col("peak_hour_calls").cast(LongType())         .alias("peak_hour_calls"),
            col("avg_revenue_per_min").cast(DoubleType())   .alias("avg_revenue_per_min"),
            col("total_events").cast(LongType())            .alias("total_events"),
            col("network_drops").cast(LongType())           .alias("network_drops"),
            col("tower_outages").cast(LongType())           .alias("tower_outages"),
            col("critical_events").cast(LongType())         .alias("critical_events"),
            col("avg_signal_strength").cast(DoubleType())   .alias("avg_signal_strength"),
            col("avg_data_speed").cast(DoubleType())        .alias("avg_data_speed"),
            col("towers_used").cast(LongType())             .alias("towers_used"),
            col("drop_rate_pct").cast(DoubleType())         .alias("drop_rate_pct"),
            col("missed_rate_pct").cast(DoubleType())       .alias("missed_rate_pct"),
            col("avg_charge_per_call").cast(DoubleType())   .alias("avg_charge_per_call"),
            col("churn_risk_score").cast(DoubleType())      .alias("churn_risk_score"),
            col("churn_risk_label"),
            col("is_high_value").cast(IntegerType())        .alias("is_high_value"),
            col("network_health_score").cast(DoubleType())  .alias("network_health_score"),
            col("pipeline_updated_at")
        )
)

gold_count = gold_df.count()
print(f"Gold rows to merge: {gold_count:,}")

# ── Step 5: MERGE into gold Delta table ──────────────────────────────────────
print(f"\nMerging into {GOLD_TABLE}...")

delta_table = DeltaTable.forName(spark, GOLD_TABLE)

(
    delta_table.alias("existing")
        .merge(
            gold_df.alias("new"),
            """
            existing.customer_id   = new.customer_id
            AND existing.snapshot_date = new.snapshot_date
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
)

final_count = spark.sql(f"""
    SELECT COUNT(*) as cnt
    FROM {GOLD_TABLE}
    WHERE snapshot_date = '{today}'
""").collect()[0]["cnt"]

print(f"✓ Merge complete!")
print(f"  Rows merged:      {gold_count:,}")
print(f"  Total rows today: {final_count:,}")

# ── Step 6: Update pipeline_run_log is_processed = 1 ─────────────────────────
print(f"\nUpdating pipeline_run_log to is_processed=1...")
spark.sql(f"""
    UPDATE telecom.bronze.pipeline_run_log
    SET    is_processed = 1,
           logged_at    = current_timestamp()
    WHERE  run_date     = '{today}'
""")

print(f"✓ Pipeline log updated to is_processed=1 for {today}")
print(f"{'='*60}")
print(f"Gold notebook complete!")
print(f"{'='*60}")